# 02 — Capa Silver: limpieza, calidad y MERGE incremental
Solo se procesan los sismos indicados en los parámetros.
- **Acelerogramas:** se leen las cabeceras, se tipan las columnas, se filtra la señal y se calculan PGA, PGV y espectros. Los registros que no se pueden procesar van a la **cuarentena** con su motivo. Los que se procesan con avisos se conservan con la columna `problemas`.
- **Espectros y eventos:** se derivan de los registros válidos.
- Las tablas de sismos se actualizan con `MERGE INTO` (si el registro existe se actualiza, si no se inserta).
- **Maestros** (territorio, edificios, STEAD, Tainan, Manta, suelo no lineal y fragilidad): se procesan la primera vez o con `recargar_maestros = true`. Las filas descartadas van a `silver_calidad_log` y a cuarentena.

## 1. Setup — imports

In [ ]:
%pip install -q geopandas rasterio h5py pypdf pyproj shapely seaborn

In [ ]:
import io
import re
import unicodedata
import zlib
from datetime import datetime
from pathlib import Path

import geopandas as gpd
import h5py
import numpy as np
import pandas as pd
import rasterio
import shapely
from pyproj import Transformer
from rasterio.features import rasterize
from scipy import ndimage
from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import RegularGridInterpolator
from scipy.optimize import minimize
from scipy.signal import butter, cont2discrete, detrend, lfilter, sosfiltfilt, ss2tf
from scipy.signal.windows import tukey
from scipy.spatial import cKDTree
from scipy.stats import kurtosis, norm
from sklearn.ensemble import HistGradientBoostingRegressor

from pyspark.sql import functions as F, types as T
from pyspark.sql.window import Window

## 2. Catálogo, esquema y parámetros

In [ ]:
catalogo = "grupo06proyectofinal"
esquema = "sismos"
volumen = f"/Volumes/{catalogo}/{esquema}"

spark.sql(f"USE CATALOG {catalogo}")
spark.sql(f"USE SCHEMA {esquema}")
spark.conf.set("spark.sql.session.timeZone", "America/Lima")

dbutils.widgets.text("fecha_desde", "2007-01-01")
dbutils.widgets.text("fecha_hasta", "2025-12-31")
dbutils.widgets.text("recargar_maestros", "false")
fecha_desde, fecha_hasta = dbutils.widgets.get("fecha_desde"), dbutils.widgets.get("fecha_hasta")
recargar_maestros = dbutils.widgets.get("recargar_maestros").lower() == "true"

In [ ]:
PROVINCIAS_ESTUDIO = ["LIMA", "CALLAO"]
DISTRITOS_ESTUDIO = {"150104": "BARRANCO", "150122": "MIRAFLORES", "150130": "SAN BORJA", "150131": "SAN ISIDRO",
                     "150140": "SANTIAGO DE SURCO", "150141": "SURQUILLO"}
ZONA_E030_LIMA = 4
EPSG_UTM_LIMA = 32718
SEED = 700
FILTRO_HZ = (0.1, 25.0)
AMORTIGUAMIENTO = 0.05
PERIODOS = np.round(np.geomspace(0.05, 4.0, 50), 4)
G_CM_S2 = 980.665

## 3. Funciones

In [ ]:
def a_spark(pdf, esquema=None):
    pdf = pdf.copy()
    for c in pdf.columns:
        if isinstance(pdf[c].dtype, (pd.StringDtype, pd.ArrowDtype)):
            pdf[c] = pdf[c].astype(object)
        if pdf[c].dtype == object:
            pdf[c] = pdf[c].map(lambda v: v if v is None or isinstance(v, (str, bytes, list, dict, np.ndarray))
                                else (None if pd.isna(v) else str(v)))
    return spark.createDataFrame(pdf, esquema) if esquema is not None else spark.createDataFrame(pdf)


def escribir(df, tabla, particion=None, modo="overwrite", contar=True):
    escritor = df.write.format("delta").mode(modo)
    if modo == "overwrite":
        escritor = escritor.option("overwriteSchema", "true")
    if particion:
        escritor = escritor.partitionBy(particion)
    escritor.saveAsTable(tabla)
    if contar:
        n = spark.table(tabla).count()
        print(f"  {tabla}: {n:,} filas")
        return n


def subir_pandas(lotes, tabla, esquema=None, preparar=None, particion=None, mb_lote=64):
    if isinstance(lotes, pd.DataFrame):
        lotes = [lotes]
    modo, n_lotes = "overwrite", 0
    for pdf in lotes:
        if len(pdf) == 0:
            continue
        partes = max(1, int(np.ceil(pdf.memory_usage(deep=True).sum() / 1e6 / mb_lote)))
        for idx in np.array_split(np.arange(len(pdf)), partes):
            sdf = a_spark(pdf.iloc[idx], esquema)
            if preparar is not None:
                sdf = preparar(sdf)
            escribir(sdf, tabla, particion, modo, contar=False)
            modo, n_lotes = "append", n_lotes + 1
    n = spark.table(tabla).count()
    print(f"  {tabla}: {n:,} filas ({n_lotes} lotes)")
    return n


def inventario(prefijo):
    filas = []
    for t in spark.sql("SHOW TABLES").collect():
        if t.isTemporary or not t.tableName.startswith(prefijo):
            continue
        d = spark.sql(f"DESCRIBE DETAIL {t.tableName}").first()
        filas.append({"tabla": t.tableName, "filas": spark.table(t.tableName).count(), "mb": round((d["sizeInBytes"] or 0) / 1e6, 1)})
    return pd.DataFrame(filas)

In [ ]:
ESQUEMA_PROBLEMAS = "fuente string, regla string, accion string, registro_id string, detalle string, n_filas long"


def registrar_calidad(df_problemas, tabla, condicion=None):
    df = (df_problemas
          .withColumn("tabla", F.lit(tabla))
          .withColumn("fecha", F.current_timestamp())
          .select("fuente", "tabla", "regla", "accion", "registro_id", "detalle",
                  F.col("n_filas").cast("long").alias("n_filas"), "fecha"))
    filtro = f"tabla = '{tabla}'" + (f" AND {condicion}" if condicion else "")
    if spark.catalog.tableExists("silver_calidad_log"):
        df.write.format("delta").mode("overwrite").option("replaceWhere", filtro).saveAsTable("silver_calidad_log")
    else:
        df.write.format("delta").mode("overwrite").partitionBy("tabla").saveAsTable("silver_calidad_log")
    resumen = (spark.table("silver_calidad_log").where(F.col("tabla") == tabla)
               .groupBy("regla", "accion").agg(F.sum("n_filas").alias("filas")).orderBy(F.desc("filas")).collect())
    for r in resumen:
        print(f"  calidad_log [{tabla}] {r['accion']:<11} {r['regla']}: {r['filas']:,}")


def problemas(filas):
    pdf = pd.DataFrame(filas, columns=["fuente", "regla", "accion", "registro_id", "detalle", "n_filas"])
    pdf["n_filas"] = pdf["n_filas"].astype("int64")
    return a_spark(pdf, ESQUEMA_PROBLEMAS)


def merge_silver(df, tabla, claves, particion=None):
    if spark.catalog.tableExists(tabla):
        df.createOrReplaceTempView("nuevo")
        condicion = " AND ".join(f"d.{k} = n.{k}" for k in claves)
        spark.sql(f"MERGE INTO {tabla} d USING nuevo n ON {condicion} WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *")
    else:
        escribir(df, tabla, particion, contar=False)
    print(tabla, spark.table(tabla).count())

In [ ]:
VS30_RELLENO_MAR = 600.0


def preparar_vs30():
    px = spark.table("bronze_vs30_pixeles").select("fila", "col", "lon", "lat", "vs30").toPandas()
    malla = px.pivot_table(index="fila", columns="col", values="vs30").sort_index().sort_index(axis=1)
    v = malla.to_numpy(float)
    mar = np.isclose(v, VS30_RELLENO_MAR) | ~np.isfinite(v)
    v = v[tuple(ndimage.distance_transform_edt(mar, return_distances=False, return_indices=True))]
    lat = px.groupby("fila").lat.first().loc[malla.index].to_numpy()
    lon = px.groupby("col").lon.first().loc[malla.columns].to_numpy()
    orden = np.argsort(lat)
    f = RegularGridInterpolator((lat[orden], lon), v[orden], bounds_error=False, fill_value=None)
    filas, cols = np.meshgrid(malla.index.to_numpy(), malla.columns.to_numpy(), indexing="ij")
    pixeles = pd.DataFrame({"fila": filas.ravel().astype("int64"), "col": cols.ravel().astype("int64"),
                            "vs30_corregido": v.ravel(), "relleno_mar": mar.ravel()})
    return pixeles, lambda lon_, lat_: f(np.column_stack([np.asarray(lat_, float), np.asarray(lon_, float)]))

In [ ]:
try:
    spark.sql("SELECT st_contains(st_geomfromwkt('POLYGON((0 0,1 0,1 1,0 1,0 0))', 4326), st_point(0.5, 0.5, 4326)) AS ok").first()
    USAR_ST = True
except Exception:
    USAR_ST = False


def unir_poligonos(puntos, poligonos, atributos, clave, lon="lon", lat="lat"):
    pol = poligonos[atributos + ["geometry_wkt"]].rename(columns={"geometry_wkt": "_pol_wkt"})
    pol = pol[pol["_pol_wkt"].notna()].reset_index(drop=True)
    pol["_pol_wkt"] = pol["_pol_wkt"].astype(object)
    pol["_pid"] = np.arange(len(pol))
    geoms = shapely.from_wkt(pol["_pol_wkt"].to_numpy())
    pts = puntos.select(*clave, F.col(lon).alias("_lon"), F.col(lat).alias("_lat"))
    if USAR_ST:
        pol["_xmin"], pol["_ymin"], pol["_xmax"], pol["_ymax"] = shapely.bounds(geoms).T
        cond = (F.col("_lon").between(F.col("_xmin"), F.col("_xmax")) & F.col("_lat").between(F.col("_ymin"), F.col("_ymax"))
                & F.expr("st_contains(st_geomfromwkt(_pol_wkt, 4326), st_point(_lon, _lat, 4326))"))
        cruces = pts.join(F.broadcast(a_spark(pol)), cond, "inner").select(*clave, "_pid", *atributos)
    else:
        pdf = pts.toPandas()
        i_pt, i_pol = shapely.STRtree(geoms).query(shapely.points(pdf["_lon"].to_numpy(), pdf["_lat"].to_numpy()),
                                                   predicate="within")
        cr = pdf.iloc[i_pt][clave].reset_index(drop=True)
        cr["_pid"] = i_pol
        cr = cr.join(pol.loc[i_pol, atributos].reset_index(drop=True))
        cruces = a_spark(cr)
    primero = Window.partitionBy(*clave).orderBy("_pid")
    cruces = cruces.withColumn("_r", F.row_number().over(primero)).where("_r = 1").drop("_r", "_pid")
    return puntos.join(cruces, clave, "left")

In [ ]:
_CACHE_SDOF = {}


def _filtro_sdof(periodo, dt, z):
    if (periodo, dt, z) in _CACHE_SDOF:
        return _CACHE_SDOF[(periodo, dt, z)]
    w = 2 * np.pi / periodo
    A = np.array([[0.0, 1.0], [-w * w, -2 * z * w]])
    B = np.array([[0.0], [-1.0]])
    C = np.array([[1.0, 0.0]])
    D = np.array([[0.0]])
    Ad, Bd, Cd, Dd, _ = cont2discrete((A, B, C, D), dt, method="foh")
    b, a = ss2tf(Ad, Bd, Cd, Dd)
    _CACHE_SDOF[(periodo, dt, z)] = (b[0], a, w)
    return _CACHE_SDOF[(periodo, dt, z)]


def espectro_sa(acc_g, dt, periodos=PERIODOS, z=AMORTIGUAMIENTO):
    sa = np.empty(len(periodos))
    for k, periodo in enumerate(periodos):
        b, a, w = _filtro_sdof(float(periodo), float(dt), z)
        sa[k] = w * w * np.abs(lfilter(b, a, acc_g)).max()
    return sa


def filtrar_cms2(acc_cms2, fs):
    a = detrend(acc_cms2 - acc_cms2.mean(), type="linear")
    a = a * tukey(len(a), 0.05)
    sos = butter(4, FILTRO_HZ, btype="bandpass", fs=fs, output="sos")
    return sosfiltfilt(sos, a)


def procesar_componente(acc_cms2, fs):
    dt = 1.0 / fs
    a = filtrar_cms2(acc_cms2, fs)
    vel = cumulative_trapezoid(a, dx=dt, initial=0.0)
    acc_g = a / G_CM_S2
    return acc_g, np.abs(acc_g).max(), np.abs(vel).max(), espectro_sa(acc_g, dt)


def intensidad_arias(acc_g, fs):
    return np.pi / (2 * 9.80665) * np.cumsum((acc_g * 9.80665) ** 2) / fs


def duracion_d595(arias, fs):
    return (np.searchsorted(arias, 0.95 * arias[-1]) - np.searchsorted(arias, 0.05 * arias[-1])) / fs


def variables_onda(x1, x2, fs):
    k = min(len(x1), len(x2))
    pgas, pgvs, sas, arias, d595 = [], [], [], [], []
    for x in (x1[:k], x2[:k]):
        acc_g, pga, pgv, sa = procesar_componente(np.asarray(x, float), fs)
        ia = intensidad_arias(acc_g, fs)
        pgas.append(pga); pgvs.append(pgv); sas.append(sa); arias.append(ia[-1]); d595.append(duracion_d595(ia, fs))
    sa_geo = np.sqrt(sas[0] * sas[1])
    return {"pga_geo_g": float(np.sqrt(pgas[0] * pgas[1])), "pgv_geo_cms": float(np.sqrt(pgvs[0] * pgvs[1])),
            "tp_s": float(PERIODOS[int(sa_geo.argmax())]), "sa_max_geo_g": float(sa_geo.max()), "sa_geo": sa_geo.tolist(),
            "arias_ms": float(np.sqrt(arias[0] * arias[1])), "d595_s": float(np.sqrt(d595[0] * d595[1]))}

## 4. Sismos a procesar

In [ ]:
MAESTROS = Path(f"{volumen}/landing/maestros")
eventos = sorted(r[0] for r in spark.table("bronze_cismid_registros").select("_evento_carga").distinct().collect()
                 if fecha_desde <= r[0][:10] <= fecha_hasta)
condicion_eventos = "evento IN (" + (", ".join(f"'{e}'" for e in eventos) or "''") + ")"
condicion_calidad = "(" + (" OR ".join(f"registro_id LIKE '{e}/%'" for e in eventos) or "FALSE") + ")"
print("Sismos a procesar:", eventos)

## 5. Acelerogramas: lectura de cabeceras y procesamiento de la señal

In [ ]:
SECCIONES = {
    "estacion": "INFORMATION ABOUT THE SEISMIC STATION",
    "sismo": "INFORMATION ABOUT THE EARTHQUAKE",
    "registro": "INFORMATION ABOUT THE RECORD",
    "datos": "ACCELERATION DATA",
}
NOMBRE_COMPONENTE = {"EO": "EW", "EW": "EW", "E": "EW", "NS": "NS", "N": "NS", "UD": "UD", "Z": "UD", "V": "UD"}


def _num(texto):
    m = re.search(r"-?\d+(?:\.\d+)?", texto or "")
    return float(m.group()) if m else None


def parsear_cismid(texto):
    lineas = [re.sub(r"^#\s?", "", l).rstrip() for l in texto.splitlines()]
    idx = {k: next(i for i, l in enumerate(lineas) if v in l.upper()) for k, v in SECCIONES.items()}

    def campos(a, b):
        d = {}
        for l in lineas[a + 1:b]:
            if ":" in l:
                k, v = l.split(":", 1)
                d.setdefault(re.sub(r"\s+", " ", k.strip()).upper(), v.strip())
        return d

    est = campos(idx["estacion"], idx["sismo"])
    sis = campos(idx["sismo"], idx["registro"])
    reg = campos(idx["registro"], idx["datos"])

    if "STATION CODE" in est:
        codigo, nombre = est["STATION CODE"].split()[0], est.get("STATION NAME", "").strip("﻿ ")
    else:
        st = est.get("STATION", "")
        codigo = st.split()[0] if st else None
        nombre = re.search(r"\((.*)\)", st).group(1) if "(" in st else st
    if "COORDINATES" in est:
        lat, lon = [float(x) for x in est["COORDINATES"].split(",")[:2]]
    else:
        lat, lon = _num(est.get("LATITUDE")), _num(est.get("LONGITUDE"))

    fecha_txt = sis.get("DATE (LOCAL)") or sis.get("DATE", "")
    try:
        fecha = datetime.strptime(fecha_txt.strip(), "%Y-%m-%d").date().isoformat()
    except ValueError:
        fecha = datetime.strptime(re.sub(r"\s+", " ", fecha_txt.strip()), "%B %d, %Y").date().isoformat()
    mag_txt = sis.get("MAGNITUDE", "")
    tipo_mag = re.search(r"\b(Mw|ML|Ms|mb|M)\b", mag_txt, re.I)
    max_txt = reg.get("MAXIMUM ACCELERATION") or reg.get("MAX. ACC. (FILTERED)") or ""
    max_cab = [float(x) for x in re.findall(r"-?\d+\.?\d*", re.sub(r"\([^)]*\)", "", max_txt))]

    cab = {
        "estacion": codigo, "estacion_nombre": nombre, "red": est.get("NETWORK"),
        "lat": lat, "lon": lon,
        "fs": _num(est.get("SAMPLING FREQUENCY (HZ)")),
        "fecha_evento": fecha,
        "hora_local": (sis.get("ORIGIN TIME (LOCAL)") or "").split()[0] if sis.get("ORIGIN TIME (LOCAL)") else None,
        "ev_lat": _num(sis.get("LATITUDE")), "ev_lon": _num(sis.get("LONGITUDE")),
        "ev_prof_km": _num(sis.get("DEPTH (KM)")),
        "magnitud": _num(mag_txt), "magnitud_tipo": tipo_mag.group(1) if tipo_mag else None,
        "fuente_sismo": sis.get("INFORMATION SOURCE"),
        "inicio_registro": reg.get("RECORD TIME (UTC-0)") or reg.get("START_TIME (UTC-0)") or reg.get("RECORD TIME (LOCAL)"),
        "n_cabecera": int(_num(reg.get("NUMBER OF SAMPLES") or reg.get("NUMBER DATA")) or 0),
        "unidades": reg.get("DATA UNITS"),
        "pga_cabecera_cms2": max(abs(x) for x in max_cab) if max_cab else None,
    }

    nombres = lineas[idx["datos"] + 1].split()
    filas = [l for l in lineas[idx["datos"] + 2:] if l.strip()]
    datos = pd.DataFrame(np.array([l.split() for l in filas], dtype=float), columns=nombres)
    datos = datos.rename(columns=lambda c: NOMBRE_COMPONENTE.get(c.upper(), c.upper()))
    return cab, datos[["EW", "NS", "UD"]]


def procesar_registro(evento, archivo, texto):
    fila = {"evento": evento, "archivo_estacion": archivo, "problemas": "", "procesado": False}
    avisos = []
    try:
        cab, datos = parsear_cismid(texto)
    except Exception as e:
        fila["problemas"] = f"cabecera_ilegible: {type(e).__name__}"
        return fila
    fila.update(cab)
    n = len(datos)
    fila["n_muestras"] = n
    if not cab["fs"]:
        fila["problemas"] = "sin_frecuencia_muestreo"
        return fila
    fila["duracion_s"] = n / cab["fs"]
    if cab["n_cabecera"] and cab["n_cabecera"] != n:
        avisos.append(f"n_muestras {n} != cabecera {cab['n_cabecera']}")
    if (cab["unidades"] or "").replace(" ", "").lower() not in ("cm/s2", "cm/s^2", "gal"):
        avisos.append(f"unidades '{cab['unidades']}'")
    if cab["lat"] is None or cab["lon"] is None:
        avisos.append("sin_coordenadas_estacion")
    if datos.isna().any().any() or n < 10 * cab["fs"]:
        fila["problemas"] = "datos_incompletos_o_registro_menor_a_10s"
        return fila

    pga_crudo = (datos - datos.mean()).abs().max().max()
    if cab["pga_cabecera_cms2"]:
        fila["pga_cabecera_dif_pct"] = 100 * (pga_crudo - cab["pga_cabecera_cms2"]) / cab["pga_cabecera_cms2"]
        if abs(fila["pga_cabecera_dif_pct"]) > 10:
            avisos.append(f"PGA difiere {fila['pga_cabecera_dif_pct']:.0f}% de la cabecera")

    for comp in ["EW", "NS", "UD"]:
        _, pga, pgv, sa = procesar_componente(datos[comp].to_numpy(float), cab["fs"])
        fila[f"pga_{comp.lower()}_g"], fila[f"pgv_{comp.lower()}_cms"], fila[f"sa_{comp.lower()}"] = pga, pgv, sa.tolist()
    fila["pga_geo_g"] = float(np.sqrt(fila["pga_ew_g"] * fila["pga_ns_g"]))
    fila["pgv_geo_cms"] = float(np.sqrt(fila["pgv_ew_cms"] * fila["pgv_ns_cms"]))
    sa_geo = np.sqrt(np.array(fila["sa_ew"]) * np.array(fila["sa_ns"]))
    fila["sa_geo"] = sa_geo.tolist()
    fila["tp_s"] = float(PERIODOS[int(sa_geo.argmax())])
    fila["sa_max_geo_g"] = float(sa_geo.max())
    fila["problemas"] = "; ".join(avisos)
    fila["procesado"] = True
    return fila

In [ ]:
ESQUEMA_REGISTRO = T.StructType([
    T.StructField("evento", T.StringType()), T.StructField("archivo_estacion", T.StringType()),
    T.StructField("estacion", T.StringType()), T.StructField("estacion_nombre", T.StringType()),
    T.StructField("red", T.StringType()),
    T.StructField("lat", T.DoubleType()), T.StructField("lon", T.DoubleType()),
    T.StructField("fs", T.DoubleType()), T.StructField("n_muestras", T.LongType()),
    T.StructField("duracion_s", T.DoubleType()), T.StructField("unidades", T.StringType()),
    T.StructField("fecha_evento", T.StringType()), T.StructField("hora_local", T.StringType()),
    T.StructField("ev_lat", T.DoubleType()), T.StructField("ev_lon", T.DoubleType()),
    T.StructField("ev_prof_km", T.DoubleType()), T.StructField("magnitud", T.DoubleType()),
    T.StructField("magnitud_tipo", T.StringType()), T.StructField("fuente_sismo", T.StringType()),
    T.StructField("inicio_registro", T.StringType()),
    T.StructField("pga_cabecera_cms2", T.DoubleType()), T.StructField("pga_cabecera_dif_pct", T.DoubleType()),
    *[T.StructField(f"pga_{c}_g", T.DoubleType()) for c in ["ew", "ns", "ud", "geo"]],
    *[T.StructField(f"pgv_{c}_cms", T.DoubleType()) for c in ["ew", "ns", "ud", "geo"]],
    T.StructField("tp_s", T.DoubleType()), T.StructField("sa_max_geo_g", T.DoubleType()),
    *[T.StructField(f"sa_{c}", T.ArrayType(T.DoubleType())) for c in ["ew", "ns", "ud", "geo"]],
    T.StructField("problemas", T.StringType()), T.StructField("procesado", T.BooleanType()),
])
COLUMNAS = [f.name for f in ESQUEMA_REGISTRO.fields]


cismid_b = spark.table("bronze_cismid_registros").where(F.col("_evento_carga").isin(eventos))
filas = []
for ev in eventos:
    lote = cismid_b.where(F.col("evento") == ev).select("evento", "archivo_estacion", "contenido").toPandas()
    filas += [procesar_registro(r.evento, r.archivo_estacion, r.contenido) for r in lote.itertuples()]
    print(f"  {ev}: {len(lote)} registros")
reg_pdf = pd.DataFrame(filas).reindex(columns=COLUMNAS)
reg_pdf["n_muestras"] = reg_pdf["n_muestras"].astype("Int64")
for c in ["sa_ew", "sa_ns", "sa_ud", "sa_geo"]:
    reg_pdf[c] = reg_pdf[c].astype(object).where(reg_pdf[c].notna(), None)
registros = a_spark(reg_pdf, ESQUEMA_REGISTRO)
print(f"Registros procesados: {len(reg_pdf)}")
registros.groupBy("evento", "procesado").count().orderBy("evento").show(truncate=False)

## 6. Vs30 de cada estación y distancia al sismo

In [ ]:
_, vs30_en = preparar_vs30()
est_vs = registros.select("evento", "archivo_estacion", "lon", "lat").toPandas()
est_vs["vs30"] = vs30_en(est_vs.lon, est_vs.lat)
est_vs = a_spark(est_vs[["evento", "archivo_estacion", "vs30"]], "evento string, archivo_estacion string, vs30 double")

R_TIERRA = 6371.0
lat1, lat2 = F.radians("lat"), F.radians("ev_lat")
dist_epi = 2 * R_TIERRA * F.asin(F.sqrt(F.sin((lat2 - lat1) / 2) ** 2
                                         + F.cos(lat1) * F.cos(lat2) * F.sin((F.radians("ev_lon") - F.radians("lon")) / 2) ** 2))

registros_vs = (registros
    .join(est_vs, ["evento", "archivo_estacion"], "left")
    .withColumn("vs30", F.nanvl("vs30", F.lit(None).cast("double")))
    .withColumn("dist_epi_km", dist_epi)
    .withColumn("dist_hipo_km", F.sqrt(F.col("dist_epi_km") ** 2 + F.col("ev_prof_km") ** 2))
    .withColumn("periodos_s", F.array(*[F.lit(float(p)) for p in PERIODOS])))

## 7. MERGE de registros, cuarentena y registro de calidad

In [ ]:
validos = (registros_vs.where("procesado").drop("procesado")
           .withColumn("problemas", F.when(F.col("vs30").isNull(), F.concat_ws("; ", F.col("problemas"), F.lit("estacion_sin_vs30")))
                                     .otherwise(F.col("problemas"))))
merge_silver(validos, "silver_ondas_registros", ["evento", "archivo_estacion"], particion="evento")

rechazados = registros_vs.where("NOT procesado").select("evento", "archivo_estacion", "estacion", F.col("problemas").alias("motivo_rechazo"))
if spark.catalog.tableExists("silver_ondas_registros_cuarentena"):
    rechazados.write.mode("overwrite").option("replaceWhere", condicion_eventos).saveAsTable("silver_ondas_registros_cuarentena")
else:
    rechazados.write.saveAsTable("silver_ondas_registros_cuarentena")
print("silver_ondas_registros_cuarentena", rechazados.count())

calidad = (registros_vs
    .withColumn("problemas", F.when(F.col("vs30").isNull() & F.col("procesado"),
                                    F.concat_ws("; ", F.col("problemas"), F.lit("estacion_sin_vs30")))
                              .otherwise(F.col("problemas")))
    .where((~F.col("procesado")) | (F.col("problemas") != ""))
    .select(F.lit("cismid").alias("fuente"),
            F.when(F.col("procesado"), "advertencia_registro").otherwise("registro_no_procesable").alias("regla"),
            F.when(F.col("procesado"), "advertencia").otherwise("descartado").alias("accion"),
            F.concat_ws("/", "evento", "archivo_estacion").alias("registro_id"),
            F.col("problemas").alias("detalle"), F.lit(1).alias("n_filas")))
registrar_calidad(calidad, "ondas_registros", condicion_calidad)

## 8. Espectros y eventos

In [ ]:
validos = spark.table("silver_ondas_registros").where(F.col("evento").isin(eventos))
partes = []
for comp in ["ew", "ns", "ud", "geo"]:
    partes.append(validos.select(
        "evento", "estacion", "archivo_estacion", "lat", "lon", "vs30", "dist_hipo_km", "periodos_s",
        F.lit(comp.upper()).alias("componente"),
        F.posexplode(f"sa_{comp}").alias("indice_periodo", "Sa")))
espectros = (partes[0].unionByName(partes[1]).unionByName(partes[2]).unionByName(partes[3])
             .select("evento", "estacion", "archivo_estacion", "componente", "lat", "lon", "vs30", "dist_hipo_km",
                     "indice_periodo", F.element_at("periodos_s", F.col("indice_periodo") + 1).alias("T"), "Sa"))
merge_silver(espectros, "silver_espectros", ["evento", "archivo_estacion", "componente", "indice_periodo"], particion="evento")

eventos_df = (validos.groupBy("evento").agg(
        F.first("fecha_evento").alias("fecha"), F.first("hora_local").alias("hora_local"),
        F.first("ev_lat").alias("lat"), F.first("ev_lon").alias("lon"), F.first("ev_prof_km").alias("profundidad_km"),
        F.first("magnitud").alias("magnitud_cabecera"), F.first("magnitud_tipo").alias("magnitud_tipo"),
        F.count("*").alias("n_estaciones"),
        F.max("pga_geo_g").alias("pga_geo_max_g"), F.expr("percentile_approx(pga_geo_g, 0.5)").alias("pga_geo_mediana_g"),
        F.min("dist_hipo_km").alias("dist_hipo_min_km"))
    .withColumn("magnitud_catalogo", F.regexp_extract("evento", r"_M[wl]?(\d+\.\d+)$", 1).cast("double")))
merge_silver(eventos_df, "silver_eventos", ["evento"])
display(spark.table("silver_eventos").orderBy("fecha").toPandas())

## 9. Maestros
Se procesan la primera vez o cuando `recargar_maestros = true`.

In [ ]:
procesar_maestros = recargar_maestros or not spark.catalog.tableExists("silver_edificios")
print("Procesar maestros:", procesar_maestros)

### Territorio y edificios de los 6 distritos

In [ ]:
if procesar_maestros:
    CONFIANZA_MIN = 0.65
    ALTURA_POR_PISO_M = 3.0
    FRACCION_ALTURA_ESTABLE = 0.75
    PESO_REFERENCIA_WEB = 20
    ALTURA_CORTE_TORRE_M = 40.0
    VENTANA_VECINDARIO_PX = 15
    OSM_MIN_PISOS_CONTRASTE = 5
    OSM_PISOS_SATURACION = 18
    H3_RES_EDIFICIO = 9
    H3_RES_SUELO = 8

In [ ]:
if procesar_maestros:
    lim = spark.table("bronze_limites_inei").where(F.col("NOMBPROV").isin(PROVINCIAS_ESTUDIO)).toPandas()
    lim["geometria_fuente"] = "INEI"
    if spark.catalog.tableExists("bronze_limites_osm_distritos"):
        osm = spark.table("bronze_limites_osm_distritos").select("ubigeo", "nombre", "geometry_wkt").toPandas()
        sin_tilde = lambda t: unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode().upper().replace("-", " ")
        inei = lim.set_index("IDDIST")
        lim = pd.DataFrame({"IDDIST": osm.ubigeo, "NOMBDIST": osm.nombre.map(sin_tilde),
                            "NOMBPROV": osm.ubigeo.map(inei.NOMBPROV).fillna(osm.ubigeo.str[:2].map({"15": "LIMA", "07": "CALLAO"})),
                            "NOMBDEP": osm.ubigeo.map(inei.NOMBDEP).fillna(osm.ubigeo.str[:2].map({"15": "LIMA", "07": "CALLAO"})),
                            "geometry_wkt": osm.geometry_wkt, "geometria_fuente": "OSM"})
    elif spark.catalog.tableExists("bronze_limites_complemento_osm"):
        comp = spark.table("bronze_limites_complemento_osm").select("IDDIST", "geometry_wkt", "fuente").toPandas()
        falta = lim["geometry_wkt"].isna() & lim["IDDIST"].isin(comp["IDDIST"])
        lim.loc[falta, "geometry_wkt"] = lim.loc[falta, "IDDIST"].map(dict(zip(comp["IDDIST"], comp["geometry_wkt"])))
        lim.loc[falta, "geometria_fuente"] = lim.loc[falta, "IDDIST"].map(dict(zip(comp["IDDIST"], comp["fuente"])))
    sin_geometria = lim.loc[lim["geometry_wkt"].isna(), "NOMBDIST"].tolist()
    lim = lim[lim["geometry_wkt"].notna()].reset_index(drop=True)
    gdf = gpd.GeoDataFrame(lim, geometry=gpd.GeoSeries.from_wkt(lim["geometry_wkt"].astype(object)), crs=4326)
    invalidas = (~gdf.is_valid).sum()
    gdf["geometry"] = gdf.geometry.make_valid()
    completados = gdf["geometria_fuente"] != "INEI"
    if completados.any() and not completados.all():
        area_antes = gdf.to_crs(EPSG_UTM_LIMA).area
        gdf.loc[~completados, "geometry"] = gdf.loc[~completados].geometry.difference(gdf.loc[completados].geometry.union_all())
        recorte = (area_antes - gdf.to_crs(EPSG_UTM_LIMA).area) / 1e6
        print("Completados con OSM:", gdf.loc[completados, "NOMBDIST"].tolist(),
              "| área recortada a vecinos (km²):", {d: round(r, 2) for d, r in zip(gdf["NOMBDIST"], recorte) if r > 0.01},
              "| sin geometría:", sin_geometria)
    utm = gdf.to_crs(EPSG_UTM_LIMA)
    dist = pd.DataFrame({
        "ubigeo": gdf["IDDIST"], "distrito": gdf["NOMBDIST"], "provincia": gdf["NOMBPROV"], "departamento": gdf["NOMBDEP"],
        "area_km2": (utm.area / 1e6).round(3),
        "centroide_lon": utm.centroid.to_crs(4326).x, "centroide_lat": utm.centroid.to_crs(4326).y,
        "geometry_wkt": gdf.geometry.to_wkt(), "geometria_fuente": gdf["geometria_fuente"],
        "en_estudio": gdf["IDDIST"].isin(list(DISTRITOS_ESTUDIO)),
    })
    print(f"Distritos: {len(dist)} ({dist.provincia.value_counts().to_dict()}) | geometrías reparadas: {invalidas}")
    subir_pandas(dist, "silver_distritos")
    faltan_estudio = set(DISTRITOS_ESTUDIO) - set(dist.loc[dist.en_estudio, "ubigeo"])
    assert not faltan_estudio, f"Distritos de estudio sin geometría: {faltan_estudio}"
    print("Área de estudio:", dist.loc[dist.en_estudio, "distrito"].tolist(), f"| {dist.loc[dist.en_estudio, 'area_km2'].sum():.1f} km²")

    BBOX = gdf.loc[gdf["IDDIST"].isin(list(DISTRITOS_ESTUDIO))].total_bounds
    print("BBOX del área de estudio:", BBOX.round(3))


    def asignar_distrito(df, clave):
        return unir_poligonos(df, dist, ["ubigeo", "distrito", "provincia"], clave)

In [ ]:
if procesar_maestros:
    perfiles = spark.table("bronze_e030_perfiles").drop("_fuente", "_source_file", "_ingested_at")
    factor_s = spark.table("bronze_e030_factor_suelo").where(F.col("zona") == ZONA_E030_LIMA).select("perfil", "S")
    z = spark.table("bronze_e030_zonas").where(F.col("zona") == ZONA_E030_LIMA).first()["Z_g"]
    e030 = (perfiles.join(factor_s, "perfil").withColumn("zona", F.lit(ZONA_E030_LIMA)).withColumn("Z_g", F.lit(z))
            .select("perfil", "descripcion", "vs30_min", "vs30_max", "zona", "Z_g", "S", "TP_s", "TL_s").orderBy("perfil"))
    escribir(e030, "silver_e030_parametros")
    e030 = spark.table("silver_e030_parametros")
    E030 = {r["perfil"]: r.asDict() for r in e030.collect()}
    e030.orderBy("perfil").show(truncate=False)


    def perfil_e030(col_vs30):
        return (F.when(col_vs30 > E030["S0"]["vs30_min"], "S0")
                 .when(col_vs30 > E030["S1"]["vs30_min"], "S1")
                 .when(col_vs30 >= E030["S2"]["vs30_min"], "S2")
                 .when(col_vs30.isNotNull(), "S3"))

In [ ]:
if procesar_maestros:
    px_vs30, vs30_en = preparar_vs30()
    vs = (spark.table("bronze_vs30_pixeles").select("fila", "col", "lon", "lat", F.col("vs30").alias("vs30_usgs"))
          .join(a_spark(px_vs30, "fila long, col long, vs30_corregido double, relleno_mar boolean"), ["fila", "col"], "left")
          .withColumnRenamed("vs30_corregido", "vs30"))
    vs_dist = asignar_distrito(vs, ["fila", "col"])

    e030_b = F.broadcast(e030.select("perfil", "S", "TP_s", "TL_s", "Z_g"))
    suelo = (vs_dist
             .withColumn("perfil", perfil_e030(F.col("vs30")))
             .join(e030_b, "perfil", "left")
             .withColumn("en_area_estudio", F.col("ubigeo").isin(list(DISTRITOS_ESTUDIO)))
             .withColumn("h3_r8", F.expr(f"h3_longlatash3string(lon, lat, {H3_RES_SUELO})"))
             .withColumnRenamed("TP_s", "TP").withColumnRenamed("TL_s", "TL")
             .select("fila", "col", "lon", "lat", "vs30", "vs30_usgs", "relleno_mar", "perfil", "S", "TP", "TL", "Z_g",
                     "ubigeo", "distrito", "provincia", "en_area_estudio", "h3_r8"))
    escribir(suelo, "silver_suelo")

    s = spark.table("silver_suelo")
    s.where("en_area_estudio").groupBy("perfil").agg(F.count("*").alias("pixeles"), F.round(F.avg("vs30")).alias("vs30_medio")).orderBy("perfil").show()
    registrar_calidad(problemas([
        {"fuente": "vs30_usgs", "regla": "pixel_fuera_del_area_estudio", "accion": "fuera_de_alcance", "registro_id": None,
         "detalle": "píxel del rectángulo cuyo centro no cae en los distritos de estudio; se conserva para interpolar",
         "n_filas": s.where("NOT en_area_estudio").count()},
        {"fuente": "vs30_usgs", "regla": "vs30_relleno_mar_600", "accion": "imputado", "registro_id": None,
         "detalle": "píxel con el relleno de mar de USGS (600 m/s): toma el Vs30 del píxel de tierra más cercano",
         "n_filas": s.where("relleno_mar").count()},
        {"fuente": "vs30_usgs", "regla": "vs30_nulo", "accion": "descartado", "registro_id": None,
         "detalle": "píxel sin valor de Vs30", "n_filas": s.where("vs30_usgs IS NULL").count()},
    ]), "suelo")

In [ ]:
if procesar_maestros:
    DISTRITO_PDF = {"cercado_de_lima": "LIMA", "comas": "COMAS", "la_molina": "LA MOLINA", "villa_el_salvador": "VILLA EL SALVADOR"}
    ROMANOS = ["I", "II", "III", "IV", "V"]


    def normalizar(texto):
        texto = re.sub(r"\s+", " ", texto)
        return re.sub(r"(\d)\s*([.,])\s*(\d)", r"\1.\3", texto)


    pag = spark.table("bronze_microzonificacion_paginas").select("distrito", "pagina", "texto").orderBy("distrito", "pagina").toPandas()
    zonas, periodos = [], []
    for archivo, d in pag.groupby("distrito"):
        for r in d.itertuples():
            t = normalizar(r.texto)
            for m in re.finditer(r"(\d\.\d+)\s*(?:s|seg\.?)?\s*(?:y|a|–|-|hasta)\s*(\d\.\d+)\s*(?:s|seg)\b", t):
                tmin, tmax = float(m.group(1)), float(m.group(2))
                if 0 < tmin < tmax <= 5:
                    periodos.append({"archivo_pdf": archivo, "distrito": DISTRITO_PDF[archivo], "pagina": r.pagina,
                                     "T_min_s": tmin, "T_max_s": tmax, "contexto": t[max(0, m.start() - 200): m.end() + 60]})
        texto = normalizar(" ".join(d.sort_values("pagina")["texto"]))
        marcas = list(re.finditer(r"\bZona\s+(IV|V|I{1,3})\s*[:.\-–]?\s+(?=Esta|Comprende|Corresponde|Se |En |Está|Abarca|Conformad)", texto, re.I))
        for zona in ROMANOS:
            m = next((m for m in marcas if m.group(1).upper() == zona), None)
            if not m:
                continue
            sig = next((n.start() for n in marcas if n.start() > m.start() and n.group(1).upper() != zona), m.end() + 2500)
            desc = texto[m.end(): min(sig, m.end() + 2500)].strip()
            zonas.append({
                "archivo_pdf": archivo, "distrito": DISTRITO_PDF[archivo], "zona": zona,
                "descripcion": desc,
                "perfiles_e030": sorted(set(re.findall(r"\bS[0-4]\b", desc))),
                "periodos_s": sorted({float(x.replace(",", ".")) for x in re.findall(r"(\d[.,]\d{1,2})\s*(?:s\b|seg)", desc)}),
                "menciona_licuacion": bool(re.search(r"licuac", desc, re.I)),
                "menciona_relleno": bool(re.search(r"relleno|desmonte", desc, re.I)),
            })
    micro = pd.DataFrame(zonas)
    ESQ_ZONAS = ("archivo_pdf string, distrito string, zona string, descripcion string, perfiles_e030 array<string>, "
                 "periodos_s array<double>, menciona_licuacion boolean, menciona_relleno boolean")
    subir_pandas(micro, "silver_microzonificacion_zonas", esquema=ESQ_ZONAS)
    per = (pd.DataFrame(periodos, columns=["archivo_pdf", "distrito", "pagina", "T_min_s", "T_max_s", "contexto"])
           .drop_duplicates(["distrito", "pagina", "T_min_s", "T_max_s"]))
    subir_pandas(per, "silver_microzonificacion_periodos",
                 esquema="archivo_pdf string, distrito string, pagina long, T_min_s double, T_max_s double, contexto string")
    display(micro[["distrito", "zona", "menciona_relleno", "menciona_licuacion"]].assign(descripcion=micro.descripcion.str[:90]))
    display(per[["distrito", "pagina", "T_min_s", "T_max_s"]])

In [ ]:
if procesar_maestros:
    GRUPO_MATERIAL = {"Ladrillo o bloque de cemento": "ladrillo", "Piedra o sillar con cal o cemento": "ladrillo",
                      "Adobe": "adobe", "Tapia": "adobe", "Piedra con barro": "adobe", "Quincha (caña con barro)": "quincha",
                      "Madera (pona, tornillo etc.)": "madera", "Triplay / calamina / estera": "madera", "Otro material": "madera"}
    if spark.catalog.tableExists("bronze_censo2017_paredes"):
        grupo = F.create_map(*[x for par in GRUPO_MATERIAL.items() for x in (F.lit(par[0]), F.lit(par[1]))])
        por_grupo = (spark.table("bronze_censo2017_paredes").withColumn("grupo", grupo[F.col("material")])
                     .groupBy("ubigeo", "distrito").pivot("grupo", ["ladrillo", "adobe", "quincha", "madera"]).agg(F.sum("viviendas"))
                     .fillna(0))
        total = F.col("ladrillo") + F.col("adobe") + F.col("quincha") + F.col("madera")
        tipologia_censo = (por_grupo.withColumn("viviendas", total)
                           .select("ubigeo", "distrito", "viviendas",
                                   *[(F.col(g) / F.col("viviendas")).alias(f"p_{g}") for g in ["ladrillo", "adobe", "quincha", "madera"]]))
        escribir(tipologia_censo, "silver_tipologia_censo")
        faltan = spark.table("silver_distritos").join(spark.table("silver_tipologia_censo"), "ubigeo", "left_anti").select("ubigeo", "distrito").collect()
        print("Distritos de silver.distritos sin dato del censo (se tratan como albañilería):", [r.distrito for r in faltan])
        (spark.table("silver_tipologia_censo").orderBy(F.desc("p_adobe"))
         .select("distrito", "viviendas", *[F.round(100 * F.col(f"p_{g}"), 1).alias(f"pct_{g}") for g in ["ladrillo", "adobe", "quincha", "madera"]])
         .show(12, truncate=False))
    else:
        print("Falta bronze.censo2017_paredes (Parte 1, sección 13): gold usará solo albañilería y concreto")

In [ ]:
if procesar_maestros:
    ob = spark.table("bronze_open_buildings")
    n_bronze = ob.count()
    en_bbox = ob.where(F.col("longitude").between(float(BBOX[0]), float(BBOX[2])) & F.col("latitude").between(float(BBOX[1]), float(BBOX[3])))

    base = en_bbox.select(F.col("full_plus_code").alias("id"), F.col("longitude").alias("lon"), F.col("latitude").alias("lat"),
                          F.col("area_in_meters").alias("area_m2"), F.col("confidence").alias("confianza"),
                          F.col("geometry").alias("geometry_wkt"))
    escribir(asignar_distrito(base, ["id"]), "silver_tmp_edificios_distrito")
    con_dist = spark.table("silver_tmp_edificios_distrito")
    n_bbox = con_dist.count()

    lima = con_dist.where(F.col("ubigeo").isin(list(DISTRITOS_ESTUDIO)))
    n_lima = lima.count()
    baja_conf = lima.where(F.col("confianza") < CONFIANZA_MIN)
    edif = lima.where(F.col("confianza") >= CONFIANZA_MIN)
    n_dups = edif.groupBy("id").count().where("count > 1").count()
    edif = (edif.withColumn("_r", F.row_number().over(Window.partitionBy("id").orderBy(F.desc("confianza"), "lon", "lat")))
                .where("_r = 1").drop("_r"))
    print(f"bronze: {n_bronze:,} | en rectángulo: {n_bbox:,} | en los distritos de estudio: {n_lima:,} | "
          f"confianza ≥ {CONFIANZA_MIN}: {n_lima - baja_conf.count():,} | ids duplicados: {n_dups}")

In [ ]:
if procesar_maestros:
    edif = (edif
            .withColumn("h3_id", F.expr(f"h3_longlatash3string(lon, lat, {H3_RES_EDIFICIO})"))
            .withColumn("h3_r8", F.expr(f"h3_toparent(h3_id, {H3_RES_SUELO})")))

In [ ]:
if procesar_maestros:
    teselas = spark.table("bronze_open_buildings_25d_teselas").select("tesela", "anio", "_source_file", "xmin", "ymin", "xmax", "ymax", "res_m", "nodata").toPandas()
    ANIOS_25D = sorted(int(a) for a in teselas.anio.unique())
    print(f"Teselas de altura: {len(teselas)} | años: {ANIOS_25D} | resolución: {teselas.res_m.unique()} m")

    escribir(edif.select("id", "lon", "lat", "area_m2", "geometry_wkt"), "silver_tmp_edificios_huella")
    pos = spark.table("silver_tmp_edificios_huella").toPandas()
    print(f"Edificios descargados: {len(pos):,}")
    a_utm = Transformer.from_crs(4326, EPSG_UTM_LIMA, always_xy=True)
    pos["x_utm"], pos["y_utm"] = a_utm.transform(pos["lon"].to_numpy(), pos["lat"].to_numpy())
    wkt = pos.pop("geometry_wkt").astype(object)
    huellas = shapely.transform(shapely.from_wkt(wkt.where(wkt.notna(), None).to_numpy()), lambda c: np.column_stack(a_utm.transform(c[:, 0], c[:, 1])))
    arbol_huellas = shapely.STRtree(huellas)
    x, y = pos["x_utm"].to_numpy(), pos["y_utm"].to_numpy()


    def huella_de_puntos(lon, lat, cercania_m=0.0):
        px, py = a_utm.transform(np.asarray(lon, float), np.asarray(lat, float))
        puntos = shapely.points(px, py)
        i_pt, i_h = arbol_huellas.query(puntos, predicate="within")
        if cercania_m:
            sin_huella = np.setdiff1d(np.arange(len(puntos)), i_pt)
            j_pt, j_h = arbol_huellas.query_nearest(puntos[sin_huella], max_distance=cercania_m)
            i_pt, i_h = np.r_[i_pt, sin_huella[j_pt]], np.r_[i_h, j_h]
        return i_pt, i_h


    def percentiles_huella(h, etiquetas, qs):
        m = etiquetas > 0
        lab, val = etiquetas[m], h[m]
        orden = np.lexsort((val, lab))
        lab, val = lab[orden], val[orden]
        presentes, inicio, n = np.unique(lab, return_index=True, return_counts=True)
        pos_q = inicio[:, None] + np.asarray(qs)[None, :] * (n[:, None] - 1)
        bajo = np.floor(pos_q).astype(int)
        alto = np.minimum(bajo + 1, (inicio + n - 1)[:, None])
        return presentes, val[bajo] + (val[alto] - val[bajo]) * (pos_q - bajo)


    if spark.catalog.tableExists("bronze_osm_niveles"):
        osm_pt = spark.table("bronze_osm_niveles").where(F.col("niveles").between(1, 40)).select("lon", "lat", "niveles").toPandas()
    else:
        print("Sin bronze.osm_niveles: los pisos salen solo de la altura 2.5D")
        osm_pt = pd.DataFrame({"lon": [], "lat": [], "niveles": []})
    i_pt, i_h = huella_de_puntos(osm_pt["lon"], osm_pt["lat"])
    por_huella = pd.Series(osm_pt["niveles"].to_numpy(float)[i_pt]).groupby(i_h).median()
    niveles_osm = np.full(len(pos), np.nan)
    niveles_osm[por_huella.index.to_numpy(int)] = por_huella.to_numpy()
    print(f"Edificios con pisos de OSM (punto dentro de la huella): {np.isfinite(niveles_osm).sum():,} de {len(osm_pt):,} puntos OSM")

    alt_anio = np.full((len(pos), len(ANIOS_25D)), np.nan)
    p90_anio = np.full((len(pos), len(ANIOS_25D)), np.nan)
    alt_max = np.full(len(pos), np.nan)
    centro = np.full(len(pos), np.nan)
    vecindario = np.full(len(pos), np.nan)
    tesela = np.full(len(pos), None, dtype=object)
    validas = ~shapely.is_missing(huellas)
    for t in teselas.to_dict("records"):
        idx = np.nonzero((x >= t["xmin"]) & (x < t["xmax"]) & (y > t["ymin"]) & (y <= t["ymax"]) & validas)[0]
        if not len(idx):
            continue
        with rasterio.open(MAESTROS / "open_buildings_25d" / f"{t['tesela']}.tif") as src:
            h = src.read(1).astype("float32")
            transform_t = src.transform
        h[(h == t["nodata"]) | ~np.isfinite(h)] = 0.0
        etiquetas = rasterize(zip(huellas[idx], np.arange(1, len(idx) + 1)), out_shape=h.shape, transform=transform_t, fill=0, dtype="int32")
        etiquetas[h <= 0] = 0
        presentes, pct = percentiles_huella(h, etiquetas, [0.5, 0.9, 1.0])
        k, j = idx[presentes - 1], ANIOS_25D.index(int(t["anio"]))
        alt_anio[k, j], p90_anio[k, j] = pct[:, 0], pct[:, 1]
        if int(t["anio"]) == ANIOS_25D[-1]:
            alt_max[k] = pct[:, 2]
            colp = np.clip(((x[idx] - t["xmin"]) // t["res_m"]).astype(int), 0, h.shape[1] - 1)
            filp = np.clip(((t["ymax"] - y[idx]) // t["res_m"]).astype(int), 0, h.shape[0] - 1)
            centro[idx] = h[filp, colp]
            suma = ndimage.uniform_filter(h, VENTANA_VECINDARIO_PX)[filp, colp]
            cubierto = ndimage.uniform_filter((h > 0).astype("float32"), VENTANA_VECINDARIO_PX)[filp, colp]
            vecindario[idx] = np.where(cubierto > 0, suma / np.maximum(cubierto, 1e-6), np.nan)
            tesela[idx] = t["tesela"]

    serie = pd.DataFrame(alt_anio, columns=ANIOS_25D)
    reciente = serie.iloc[:, -2:].median(axis=1)
    existe = serie.ge(FRACCION_ALTURA_ESTABLE * reciente, axis=0)
    altura_estable = serie.where(existe).median(axis=1).to_numpy()
    pos["altura_m"] = np.where(altura_estable > 0, altura_estable, np.where(centro > 0, centro, np.nan))
    pos["altura_metodo"] = np.where(altura_estable > 0, "huella_multianual", np.where(centro > 0, "centroide", "sin_altura"))
    pos["altura_ultimo_anio_m"] = serie.iloc[:, -1].to_numpy()
    pos["altura_p90_m"] = pd.DataFrame(p90_anio).where(existe.to_numpy()).median(axis=1).to_numpy()
    pos["altura_p90_ultimo_anio_m"] = p90_anio[:, -1]
    pos["altura_vecindario_m"] = vecindario
    pos["n_anios_altura"] = existe.sum(axis=1).to_numpy()
    pos["altura_max_m"] = np.where(alt_max > 0, alt_max, pos["altura_m"])
    pos["niveles_osm"], pos["tesela"] = niveles_osm, tesela
    print(pos["altura_metodo"].value_counts().to_dict())
    print("Años usados por edificio:", pos.loc[pos.altura_metodo == "huella_multianual", "n_anios_altura"].value_counts().sort_index().to_dict())

    _, vs30_en = preparar_vs30()
    pos["vs30"] = vs30_en(pos["lon"], pos["lat"])
    ruta_zonas = MAESTROS / "microzonificacion_cismid" / "zonas_cismid_utm18s_25m.tif"
    pos["zona_cismid"] = 0
    if ruta_zonas.exists():
        with rasterio.open(ruta_zonas) as src_z:
            zz, tz = src_z.read(1), src_z.transform
        cz = np.floor((pos.x_utm.to_numpy() - tz.c) / tz.a).astype(int)
        fz = np.floor((pos.y_utm.to_numpy() - tz.f) / tz.e).astype(int)
        dentro = (cz >= 0) & (cz < zz.shape[1]) & (fz >= 0) & (fz < zz.shape[0])
        pos.loc[dentro, "zona_cismid"] = zz[fz[dentro], cz[dentro]]
        print("Edificios por zona CISMID:", pos.zona_cismid.value_counts().sort_index().to_dict())
    else:
        print("Falta", ruta_zonas.as_posix(), "(ver README): el perfil de suelo sale solo del Vs30")

In [ ]:
if procesar_maestros:
    REFERENCIA_WEB = [
        ("Torre Banco de la Nación", 30, -12.087128, -77.000533, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("Edificio BBVA", 19, -12.093705, -77.021076, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("Torre Begonias", 26, -12.092114, -77.023963, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("The Westin Lima", 30, -12.091474, -77.024632, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("Torre Chocavento", 25, -12.097605, -77.020904, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("Torre Barlovento", 31, -12.097867, -77.020646, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("Torre Fórum", 30, -12.100471, -77.018711, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("Torre Panamá", 29, -12.098188, -77.019618, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("Torre Orquídeas", 27, -12.092008, -77.028473, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("T Tower", 25, -12.090926, -77.026784, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("Torre Javier Prado", 27, -12.092002, -77.028816, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("JW Marriott Lima", 25, -12.131516, -77.029504, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("Lima Central Tower", 25, -12.097544, -76.972975, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("Edificio Petroperú", 22, -12.097900, -77.024900, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("Edificio Capital", 23, -12.092674, -77.026500, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("Capital El Derby", 22, -12.097781, -76.972420, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("Edificio Alto Caral", 21, -12.095917, -77.022019, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("Torre Scotiabank", 21, -12.095145, -77.021962, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("Edificio Altavista", 22, -12.099811, -77.018983, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("Edificio Beyond", 29, -12.097128, -76.973186, "es.wikipedia.org/wiki/Anexo:Edificios_más_altos_de_Lima"),
        ("Centro de Convenciones de Lima", 10, -12.086374, -77.000574, "wikidata.org/wiki/Q18643986"),
        ("Hyatt Centric San Isidro", 10, -12.094986, -77.036343, "hotels.com/ho636542528"),
        ("Courtyard Lima Miraflores", 11, -12.123411, -77.027565, "travelweekly.com/Hotels/Lima-Peru/Courtyard-by-Marriott-Lima-Miraflores-p9447790"),
        ("Casa Andina Premium Miraflores", 18, -12.122044, -77.026590, "hotels.com/ho268714"),
        ("Pullman Lima San Isidro", 9, -12.095171, -77.037879, "successfulmeetings.com (Pullman Lima San Isidro)"),
        ("Sonesta Hotel El Olivar", 7, -12.097809, -77.035351, "northstarmeetingsgroup.com (Sonesta Hotel El Olivar)"),
        ("Belmond Miraflores Park", 12, -12.135324, -77.027303, "meetings-conventions.com (Miraflores Park, A Belmond Hotel)"),
        ("Swissôtel Lima", 18, -12.097248, -77.038190, "atrapalo.com.ar/hoteles/42366-0_hotel-swissotel-lima"),
        ("Novotel Lima San Isidro", 8, -12.096003, -77.036521, "successfulmeetings.com (Novotel Lima)"),
        ("Hilton Garden Inn Lima Miraflores", 18, -12.123770, -77.032795, "expedia.com (Hilton Garden Inn Lima Miraflores)"),
        ("Fairfield by Marriott Lima Miraflores", 12, -12.125463, -77.021657, "amimagazine.global (Fairfield By Marriott Lima Miraflores)"),
    ]
    web = pd.DataFrame(REFERENCIA_WEB, columns=["nombre", "pisos_web", "lat", "lon", "fuente"])
    i_pt, i_h = huella_de_puntos(web["lon"], web["lat"], cercania_m=25.0)
    web = web.iloc[i_pt].assign(k=i_h).drop_duplicates("nombre")
    web["huella_compartida"] = web.groupby("k")["nombre"].transform("size") > 1
    RASGOS_PISOS = ["altura_m", "altura_p90_m", "altura_ultimo_anio_m", "altura_p90_ultimo_anio_m", "altura_max_m",
                    "altura_vecindario_m", "log_area", "n_anios_altura"]
    MONOTONIA_PISOS = [1, 1, 1, 1, 1, 0, 0, 0]
    pos["log_area"] = np.log(pos["area_m2"].astype(float))
    web = web.join(pos[RASGOS_PISOS + ["altura_metodo", "x_utm", "y_utm"]], on="k")
    print(f"Referencia web: {len(REFERENCIA_WEB)} edificios | con huella: {len(web)} | en huella compartida: {web.huella_compartida.sum()}")

    osm_ref = pos.loc[(pos.altura_metodo == "huella_multianual") & pos.niveles_osm.between(1, 45),
                      RASGOS_PISOS + ["niveles_osm", "x_utm", "y_utm"]]
    web_ref = web.loc[~web.huella_compartida & (web.altura_metodo == "huella_multianual"),
                      RASGOS_PISOS + ["pisos_web", "x_utm", "y_utm"]]
    ref_pisos = pd.concat([osm_ref.rename(columns={"niveles_osm": "pisos_ref"}).assign(fuente="osm", peso=1.0),
                           web_ref.rename(columns={"pisos_web": "pisos_ref"}).assign(fuente="web", peso=float(PESO_REFERENCIA_WEB))],
                          ignore_index=True)


    def ajustar_pisos(d):
        gbt = HistGradientBoostingRegressor(loss="absolute_error", max_iter=400, learning_rate=0.05, min_samples_leaf=15,
                                            monotonic_cst=MONOTONIA_PISOS, random_state=SEED).fit(d[RASGOS_PISOS], d.pisos_ref)
        costo = lambda p: np.sum(d.peso * np.abs((d.altura_m - p[0]) / p[1] - d.pisos_ref))
        a, b = minimize(costo, [0.0, ALTURA_POR_PISO_M], method="Nelder-Mead").x
        return gbt, float(a), float(b)


    def a_pisos(h, a, b):
        return np.maximum(1, np.round((np.asarray(h, float) - a) / b))


    def predecir_pisos(modelo, d):
        gbt, a, b = modelo
        recta = d.altura_m.gt(ALTURA_CORTE_TORRE_M) | d.altura_p90_m.isna()
        return np.where(recta, a_pisos(d.altura_m, a, b), np.maximum(1, np.round(gbt.predict(d[RASGOS_PISOS]))))


    bloque = (ref_pisos.x_utm // 2000).astype(int).astype(str) + "_" + (ref_pisos.y_utm // 2000).astype(int).astype(str)
    ref_pisos["fold"] = bloque.map(lambda b: zlib.crc32(b.encode()) % 5)
    ref_pisos["pisos_antes"] = a_pisos(ref_pisos.altura_ultimo_anio_m, 0.0, ALTURA_POR_PISO_M)
    ref_pisos["pisos_calibrado"] = np.nan
    for f in range(5):
        en_f = ref_pisos.fold == f
        ref_pisos.loc[en_f, "pisos_calibrado"] = predecir_pisos(ajustar_pisos(ref_pisos[~en_f]), ref_pisos[en_f])
    modelo_pisos = ajustar_pisos(ref_pisos)
    _, A_PISO, B_PISO = modelo_pisos
    print(f"Hasta {ALTURA_CORTE_TORRE_M:.0f} m: gradient boosting monótono | sobre {ALTURA_CORTE_TORRE_M:.0f} m: pisos = (altura {-A_PISO:+.2f} m) / {B_PISO:.2f} m"
          f"  ({len(osm_ref):,} edificios de OSM + {len(web_ref)} de la web)")

    err_antes = ref_pisos.pisos_antes - ref_pisos.pisos_ref
    err_cal = ref_pisos.pisos_calibrado - ref_pisos.pisos_ref
    grupo = pd.cut(ref_pisos.pisos_ref, [0, 2, 4, 6, 9, 14, 99], labels=["1-2", "3-4", "5-6", "7-9", "10-14", "15+"])
    display(ref_pisos.assign(grupo=grupo, ea=err_antes.abs(), ec=err_cal.abs(), sa=err_antes, sc=err_cal,
                             ok_a=err_antes.abs() <= 1, ok_c=err_cal.abs() <= 1)
            .groupby(["fuente", "grupo"], observed=True)
            .agg(edificios=("ea", "size"), error_antes=("ea", "mean"), error_calibrado=("ec", "mean"), sesgo_antes=("sa", "mean"),
                 sesgo_calibrado=("sc", "mean"), pct_1piso_antes=("ok_a", "mean"), pct_1piso_calibrado=("ok_c", "mean"))
            .round(2))
    for fuente, d in ref_pisos.groupby("fuente"):
        print(f"{fuente}: error medio {(d.pisos_antes - d.pisos_ref).abs().mean():.2f} → {(d.pisos_calibrado - d.pisos_ref).abs().mean():.2f} pisos")
    web_ok = web[web.altura_m.notna()]
    display(web_ok.assign(pisos_antes=a_pisos(web_ok.altura_ultimo_anio_m, 0.0, ALTURA_POR_PISO_M), pisos_calibrado=predecir_pisos(modelo_pisos, web_ok))
            [["nombre", "pisos_web", "pisos_antes", "pisos_calibrado", "altura_ultimo_anio_m", "altura_m", "huella_compartida", "fuente"]]
            .round(1))

    con_altura_m = pos.altura_m.notna()
    pos["pisos_medidos"] = np.nan
    pos.loc[con_altura_m, "pisos_medidos"] = predecir_pisos(modelo_pisos, pos[con_altura_m])
    print("Pisos estimados:", pos.pisos_medidos.clip(upper=10).value_counts().sort_index().astype(int).to_dict(), "(10 = 10+)")
    subir_pandas(pos[["id", "x_utm", "y_utm", "tesela", "altura_m", "altura_max_m", "altura_ultimo_anio_m", "n_anios_altura", "altura_metodo",
                      "niveles_osm", "pisos_medidos", "vs30", "zona_cismid"]], "silver_tmp_edificios_altura",
                 esquema="id string, x_utm double, y_utm double, tesela string, altura_m double, altura_max_m double, altura_ultimo_anio_m double, "
                         "n_anios_altura long, altura_metodo string, niveles_osm double, pisos_medidos double, vs30 double, zona_cismid long")

In [ ]:
if procesar_maestros:
    CT_ALBANILERIA, CT_CONCRETO, PISOS_CONCRETO_TB = 60.0, 45.0, 5
    e030_suelo = F.broadcast(spark.table("silver_e030_parametros").select(F.col("perfil").alias("tipo_suelo"), "S",
                                                                     F.col("TP_s").alias("TP"), F.col("TL_s").alias("TL")))
    PERFIL_ZONA = {1: "S1", 2: "S2", 3: "S3", 4: "S3", 5: "S3"}
    perfil_zona = F.lit(None).cast("string")
    for zona_c, perfil_c in PERFIL_ZONA.items():
        perfil_zona = F.when(F.col("zona_cismid") == zona_c, F.lit(perfil_c)).otherwise(perfil_zona)
    con_altura = (edif.join(spark.table("silver_tmp_edificios_altura"), "id", "left")
                  .withColumn("tipo_suelo", F.coalesce(perfil_zona, perfil_e030(F.col("vs30"))))
                  .withColumn("perfil_fuente", F.when(perfil_zona.isNotNull(), "microzonificacion_cismid").otherwise("vs30_usgs"))
                  .join(e030_suelo, "tipo_suelo", "left"))
    con_altura = con_altura.withColumn("pisos_medidos", F.col("pisos_medidos").cast("int"))
    mediana_r9 = con_altura.groupBy("h3_id").agg(F.expr("percentile_approx(pisos_medidos, 0.5)").alias("pisos_h3_r9"))
    mediana_r8 = con_altura.groupBy("h3_r8").agg(F.expr("percentile_approx(pisos_medidos, 0.5)").alias("pisos_h3_r8"))
    osm_muy_alto = F.coalesce((F.col("niveles_osm") >= OSM_MIN_PISOS_CONTRASTE)
                              & (F.col("altura_max_m") < 0.5 * ALTURA_POR_PISO_M * F.least(F.col("niveles_osm"), F.lit(float(OSM_PISOS_SATURACION)))),
                              F.lit(False))
    osm_muy_bajo = F.coalesce(F.col("altura_m") > 3 * ALTURA_POR_PISO_M * F.col("niveles_osm") + ALTURA_POR_PISO_M, F.lit(False))
    edificios = (con_altura.join(mediana_r9, "h3_id", "left").join(mediana_r8, "h3_r8", "left")
        .withColumn("altura_imputada", F.col("altura_m").isNull())
        .withColumn("pisos_25d", F.coalesce("pisos_medidos", "pisos_h3_r9", "pisos_h3_r8", F.lit(1)).cast("int"))
        .withColumn("osm_aceptado", F.col("niveles_osm").isNotNull() & ~osm_muy_alto & ~osm_muy_bajo)
        .withColumn("pisos", F.when(F.col("osm_aceptado"), F.greatest(F.lit(1.0), F.round("niveles_osm")))
                              .otherwise(F.col("pisos_25d")).cast("int"))
        .withColumn("pisos_metodo", F.when(F.col("osm_aceptado"), "osm")
                                     .when(~F.col("altura_imputada"), "medido")
                                     .when(F.col("pisos_h3_r9").isNotNull(), "mediana_h3_r9")
                                     .when(F.col("pisos_h3_r8").isNotNull(), "mediana_h3_r8").otherwise("1_por_defecto"))
        .withColumn("altura_tb_m", F.when(F.col("osm_aceptado"), F.greatest(F.coalesce("altura_m", F.lit(0.0)), F.col("pisos") * ALTURA_POR_PISO_M))
                                    .when(F.col("altura_imputada"), F.col("pisos") * ALTURA_POR_PISO_M)
                                    .otherwise(F.greatest(F.col("altura_m"), F.lit(ALTURA_POR_PISO_M))))
        .withColumn("Tb", F.round(F.col("altura_tb_m") / F.when(F.col("pisos") >= PISOS_CONCRETO_TB, CT_CONCRETO)
                                                         .otherwise(CT_ALBANILERIA), 3))
        .withColumn("Tb_TP", F.round(F.col("Tb") / F.col("TP"), 3))
        .withColumn("con_microzonificacion", F.col("distrito").isin(list(DISTRITO_PDF.values())))
        .select("id", "ubigeo", "distrito", "provincia", "h3_id", "h3_r8", "lon", "lat", "x_utm", "y_utm",
                "area_m2", "confianza", "altura_m", "altura_max_m", "altura_metodo", "altura_imputada", "pisos", "pisos_metodo",
                "pisos_25d", "niveles_osm", "altura_ultimo_anio_m", "n_anios_altura", "Tb",
                "vs30", "zona_cismid", "tipo_suelo", "perfil_fuente", "S", "TP", "TL", "Tb_TP", "con_microzonificacion", "tesela",
                "geometry_wkt"))
    n_edif = escribir(edificios, "silver_edificios", particion="provincia")

In [ ]:
if procesar_maestros:
    e = spark.table("silver_edificios")
    agregados = problemas([
        {"fuente": "open_buildings_v3", "regla": "fuera_del_rectangulo_estudio", "accion": "fuera_de_alcance", "registro_id": None,
         "detalle": "edificio de la tesela 911 fuera del rectángulo de los distritos de estudio", "n_filas": n_bronze - n_bbox},
        {"fuente": "open_buildings_v3", "regla": "fuera_de_distritos_estudio", "accion": "fuera_de_alcance", "registro_id": None,
         "detalle": "centroide dentro del rectángulo pero fuera de los 6 distritos de estudio (ST_Within)", "n_filas": n_bbox - n_lima},
        {"fuente": "open_buildings_v3", "regla": "id_duplicado", "accion": "descartado", "registro_id": None,
         "detalle": "full_plus_code repetido; se conserva una fila", "n_filas": n_dups},
        {"fuente": "osm_niveles", "regla": "pisos_osm", "accion": "reemplazado", "registro_id": None,
         "detalle": "pisos tomados de building:levels de OSM (punto dentro de la huella)", "n_filas": e.where("pisos_metodo = 'osm'").count()},
        {"fuente": "osm_niveles", "regla": "osm_contradice_2.5D", "accion": "descartado", "registro_id": None,
         "detalle": "pisos de OSM incompatibles con la altura 2.5D; se usa la estimación 2.5D",
         "n_filas": e.where("niveles_osm IS NOT NULL AND pisos_metodo <> 'osm'").count()},
    ])
    por_fila = (baja_conf.select(F.lit("open_buildings_v3").alias("fuente"), F.lit(f"confianza<{CONFIANZA_MIN}").alias("regla"),
                                 F.lit("descartado").alias("accion"), F.col("id").alias("registro_id"),
                                 F.format_string("confianza=%.3f", "confianza").alias("detalle"), F.lit(1).alias("n_filas"))
                .unionByName(e.where("altura_imputada").select(
                    F.lit("open_buildings_25d").alias("fuente"), F.lit("sin_altura_2.5D").alias("regla"), F.lit("imputado").alias("accion"),
                    F.col("id").alias("registro_id"), F.concat(F.lit("pisos="), F.col("pisos").cast("string"), F.lit(" ("), F.col("pisos_metodo"), F.lit("); tesela="),
                             F.coalesce("tesela", F.lit("ninguna"))).alias("detalle"),
                    F.lit(1).alias("n_filas")))
                .unionByName(e.where("vs30 IS NULL").select(
                    F.lit("vs30_usgs").alias("fuente"), F.lit("edificio_sin_vs30").alias("regla"), F.lit("advertencia").alias("accion"),
                    F.col("id").alias("registro_id"), F.lit("el píxel de Vs30 no existe en la ventana").alias("detalle"), F.lit(1).alias("n_filas"))))
    registrar_calidad(agregados.unionByName(por_fila), "edificios")

    escribir(baja_conf.select("id", "lon", "lat", "area_m2", "confianza", "ubigeo", "distrito",
                              F.lit(f"confianza<{CONFIANZA_MIN}").alias("motivo_rechazo")), "silver_edificios_cuarentena")

    for tmp in ["silver_tmp_edificios_distrito", "silver_tmp_edificios_altura", "silver_tmp_edificios_huella"]:
        spark.sql(f"DROP TABLE IF EXISTS {tmp}")
    print("Tablas temporales eliminadas")

In [ ]:
if procesar_maestros:
    e = spark.table("silver_edificios")
    par = (e.where("niveles_osm IS NOT NULL")
            .select("id", "lon", "lat", "distrito", F.col("niveles_osm").alias("niveles"), "pisos_25d", "pisos", "pisos_metodo",
                    "altura_metodo", "altura_m", "altura_max_m", (F.col("pisos_metodo") == "osm").alias("osm_aceptado"))
            .toPandas())
    subir_pandas(par, "silver_validacion_pisos_osm")
    ok = par[par.osm_aceptado]
    print(f"Edificios con pisos de OSM: {len(par):,} | aceptados: {len(ok):,} | descartados (contradicen al 2.5D): {len(par) - len(ok):,}")
    dif = ok.pisos_25d - ok.niveles
    print(f"Estimación 2.5D vs OSM: pisos exactos {100 * (dif == 0).mean():.1f} % | ±1 piso {100 * (dif.abs() <= 1).mean():.1f} % | "
          f"error medio {dif.abs().mean():.2f} pisos | sesgo {dif.mean():+.2f}")
    grupo = pd.cut(ok.niveles, [0, 1, 2, 3, 4, 6, 9, 14, 40], labels=["1", "2", "3", "4", "5-6", "7-9", "10-14", "15+"])
    display(ok.assign(grupo=grupo, error=dif.abs(), sesgo=dif).groupby("grupo", observed=True)
            .agg(edificios=("id", "size"), pisos_osm=("niveles", "mean"), pisos_25d=("pisos_25d", "mean"), altura_2_5d_m=("altura_m", "mean"),
                 error_medio=("error", "mean"), sesgo=("sesgo", "mean")).round(2))
    display(pd.crosstab(ok.niveles.clip(upper=6), ok.pisos_25d.clip(upper=6), normalize="index",
                        rownames=["pisos OSM (6 = 6+)"], colnames=["pisos 2.5D"]).round(2))

### STEAD

In [ ]:
if procesar_maestros:
    FS_STEAD = 100.0
    sd = spark.table("bronze_stead_metadata")
    snr = F.split(F.regexp_replace("snr_db", r"[\[\]]", ""), r"\s+")


    def snr_db(k):
        return F.nanvl(F.col("snr_arr")[k].cast("double"), F.lit(None).cast("double"))


    stead = (sd
        .withColumn("snr_arr", F.filter(snr, lambda s: s != ""))
        .select("trace_name", "trace_category", (F.col("trace_category") == "earthquake_local").alias("es_sismo"),
                "network_code", "receiver_code", "receiver_type", "receiver_latitude", "receiver_longitude",
                "p_arrival_sample", "p_status", "p_weight", "s_arrival_sample", "s_status", "s_weight",
                (F.col("p_arrival_sample") / FS_STEAD).alias("p_segundos"), (F.col("s_arrival_sample") / FS_STEAD).alias("s_segundos"),
                ((F.col("s_arrival_sample") - F.col("p_arrival_sample")) / FS_STEAD).alias("s_menos_p_s"),
                "source_magnitude", "source_magnitude_type", "source_distance_km", "source_depth_km",
                "source_latitude", "source_longitude", "back_azimuth_deg",
                snr_db(0).alias("snr_e_db"), snr_db(1).alias("snr_n_db"), snr_db(2).alias("snr_z_db"),
                "coda_end_sample", "trace_start_time", "_archivo_hdf5"))
    fallas = stead.where(F.col("es_sismo") & (F.col("p_arrival_sample").isNull() | F.col("source_magnitude").isNull()))
    stead_ok = stead.join(fallas.select("trace_name"), "trace_name", "left_anti")
    escribir(stead_ok, "silver_stead_trazas")
    registrar_calidad(fallas.select(
        F.lit("stead").alias("fuente"), F.lit("sismo_sin_llegada_P_o_magnitud").alias("regla"), F.lit("descartado").alias("accion"),
        F.col("trace_name").alias("registro_id"), F.lit("no sirve para entrenar el detector de onda P").alias("detalle"),
        F.lit(1).alias("n_filas")), "stead_trazas")

    escribir(fallas.select("trace_name", "trace_category", "p_arrival_sample", "source_magnitude",
                           F.lit("sismo_sin_llegada_P_o_magnitud").alias("motivo_rechazo")), "silver_stead_cuarentena")

    s = spark.table("silver_stead_trazas")
    s.groupBy("trace_category").agg(F.count("*").alias("trazas"), F.round(F.avg("source_magnitude"), 2).alias("mag_media"),
                                    F.round(F.expr("percentile_approx(p_segundos, 0.5)"), 1).alias("p_mediana_s"),
                                    F.round(F.avg("snr_z_db"), 1).alias("snr_z_medio")).show()

In [ ]:
if procesar_maestros:
    N_TRAZAS = 10_000
    FS_STEAD = 100
    ANTES, DESPUES = 100, 300


    def rasgos_ventana(x):
        x = x - x.mean(0)
        mitad = len(x) // 2
        e1 = (x[:mitad] ** 2).mean(0) + 1e-12
        e2 = (x[mitad:] ** 2).mean(0) + 1e-12
        r = {}
        for i, c in enumerate("enz"):
            r[f"log_energia_{c}"] = float(np.log10((x[:, i] ** 2).mean() + 1e-12))
            r[f"log_max_{c}"] = float(np.log10(np.abs(x[:, i]).max() + 1e-12))
            r[f"curtosis_{c}"] = float(kurtosis(x[:, i]))
            r[f"log_ratio_mitades_{c}"] = float(np.log10(e2[i] / e1[i]))
        espectro = np.abs(np.fft.rfft(x[:, 2]))
        r["frec_dominante_z"] = float(np.fft.rfftfreq(len(x), 1 / FS_STEAD)[espectro.argmax()])
        r["log_z_sobre_h"] = float(np.log10(((x[:, 2] ** 2).mean() + 1e-12) / ((x[:, :2] ** 2).mean() + 1e-12)))
        return r


    def picar_sta_lta(z, sta=50, lta=500, umbral=4.0):
        z = z - z.mean()
        e = np.concatenate([[0.0], np.cumsum(z * z)])
        i = np.arange(lta, len(z) - sta)
        cociente = ((e[i + sta] - e[i]) / sta) / ((e[i] - e[i - lta]) / lta + 1e-20)
        sobre = np.nonzero(cociente > umbral)[0]
        return (i[sobre[0]] + sta) / FS_STEAD if sobre.size else np.nan


    trazas = (spark.table("silver_stead_trazas")
              .where(f"p_arrival_sample >= {ANTES + 500} AND p_arrival_sample <= {6000 - DESPUES} AND s_arrival_sample IS NOT NULL")
              .select("trace_name", "p_arrival_sample", "s_arrival_sample", "source_magnitude", "source_distance_km", "_archivo_hdf5")
              .orderBy(F.rand(1)).limit(N_TRAZAS).toPandas())
    archivo_hdf5 = MAESTROS / "stead" / "chunk2_muestra.hdf5"
    filas, picks = [], []
    rng_v = np.random.default_rng(2)
    with h5py.File(archivo_hdf5, "r") as h:
        for r in trazas.to_dict("records"):
            x = np.asarray(h["data"][r["trace_name"]], dtype="float64")
            p = int(r["p_arrival_sample"])
            s = int(r["s_arrival_sample"])
            filas.append({"trace_name": r["trace_name"], "tipo": "onda_P", "etiqueta": 1, **rasgos_ventana(x[p - ANTES: p + DESPUES])})
            filas.append({"trace_name": r["trace_name"], "tipo": "ruido_previo", "etiqueta": 0, **rasgos_ventana(x[p - ANTES - 400: p - ANTES])})
            i0 = p + DESPUES + int(rng_v.integers(0, max(1, min(6000 - 400, p + 1500) - (p + DESPUES))))
            if i0 + 400 <= 6000:
                filas.append({"trace_name": r["trace_name"], "tipo": "dentro_del_sismo", "etiqueta": 0, **rasgos_ventana(x[i0: i0 + 400])})
            if s - ANTES >= p + DESPUES and s + DESPUES <= 6000:
                filas.append({"trace_name": r["trace_name"], "tipo": "llegada_S", "etiqueta": 0, **rasgos_ventana(x[s - ANTES: s + DESPUES])})
            picks.append({"trace_name": r["trace_name"], "p_s": p / FS_STEAD, "pick_sta_lta_s": picar_sta_lta(x[:, 2]),
                          "magnitud": r["source_magnitude"], "distancia_km": r["source_distance_km"]})
    ventanas = pd.DataFrame(filas)
    picks = pd.DataFrame(picks)

    rasgos = [c for c in ventanas.columns if c not in ("trace_name", "tipo", "etiqueta")]
    canal_sin_senal = sorted(ventanas.loc[~np.isfinite(ventanas[rasgos].to_numpy(float)).all(axis=1), "trace_name"].unique())
    ventanas = ventanas[~ventanas.trace_name.isin(canal_sin_senal)].reset_index(drop=True)
    picks = picks[~picks.trace_name.isin(canal_sin_senal)].reset_index(drop=True)
    registrar_calidad(spark.createDataFrame(
        [("stead", "canal_sin_senal", "cuarentena", t, "rasgos de la ventana sin definir (curtosis nula)", 1) for t in canal_sin_senal],
        ESQUEMA_PROBLEMAS), "stead_ventanas")
    if canal_sin_senal:
        escribir(spark.table("silver_stead_trazas").where(F.col("trace_name").isin(canal_sin_senal))
                 .select("trace_name", "trace_category", "p_arrival_sample", "source_magnitude",
                         F.lit("canal_sin_senal").alias("motivo_rechazo")), "silver_stead_cuarentena", modo="append")
    print("Trazas a cuarentena por canal sin señal:", canal_sin_senal)

    picks["error_s"] = picks.pick_sta_lta_s - picks.p_s
    print(f"{len(trazas):,} trazas leídas de {archivo_hdf5.name} | ventanas por tipo:", ventanas.tipo.value_counts().to_dict())

    subir_pandas(ventanas, "silver_stead_ventanas")
    subir_pandas(picks, "silver_stead_picks")

### Edificios y ondas de Tainan y Manta

In [ ]:
if procesar_maestros:
    RADIO_TAINAN_KM = 40.0
    a_utm_tainan = Transformer.from_crs(4326, 32651, always_xy=True).transform
    numero = lambda s: pd.to_numeric(s, errors="coerce")
    vs30_ciudades = spark.table("bronze_vs30_pixeles_ciudades").select("ciudad", "lon", "lat", "vs30").toPandas()


    def vs30_ciudad(ciudad, lon, lat):
        px = vs30_ciudades[vs30_ciudades.ciudad == ciudad]
        i = cKDTree(px[["lon", "lat"]].to_numpy(float)).query(np.column_stack([lon, lat]))[1]
        v = px.vs30.to_numpy(float)[i]
        return np.where((v > 0) & (v < 3000), v, np.nan)


    def completar_edificio(d):
        d = d.copy()
        d["pisos"] = d.pisos.astype("int64")
        d["altura_m"] = d.pisos * 3.0
        d["Tb"] = np.maximum(0.05, d.altura_m / np.where(d.pisos >= 5, 45.0, 60.0)).round(3)
        d["severo"] = d.dano.isin(["S", "C"])
        d["suelo_cod"] = np.select([d.vs30 < 180, d.vs30 < 500, d.vs30 < 1500], [3.0, 2.0, 1.0], default=0.0)
        return d


    m_t = spark.table("bronze_tainan_edificios").toPandas()
    c_t = spark.table("bronze_tainan_casos").select("datahub_id", "Longitude").toPandas()
    m_t["datahub_id"], c_t["datahub_id"] = numero(m_t.datahub_id), numero(c_t.datahub_id)
    c_t = c_t.dropna(subset=["datahub_id"]).drop_duplicates("datahub_id").rename(columns={"Longitude": "lon_caso"})
    m_t = m_t.merge(c_t, on="datahub_id", how="left")
    lon_t = numero(m_t.Longitude)
    pisos_t = numero(m_t.No_Floors)
    tainan = pd.DataFrame({"ciudad": "Tainan", "id": m_t.Experiment_ID, "lat": numero(m_t.Latitude), "lon": lon_t.where(lon_t >= 100, numero(m_t.lon_caso)),
                           "dano": m_t.Structural_Damage, "pisos": pisos_t, "area_m2": numero(m_t.First_Floor_Area).fillna(numero(m_t.Total_Floor_Area) / pisos_t)})
    tainan = tainan.dropna(subset=["lat", "lon", "dano", "pisos", "area_m2"])
    tainan = tainan[tainan.dano.isin(list("NLMSC"))].copy()
    tainan["pisos"] = np.floor(tainan.pisos + 0.5).clip(lower=1)
    tainan["vs30"] = vs30_ciudad("Tainan", tainan.lon, tainan.lat)
    tainan = completar_edificio(tainan.dropna(subset=["vs30"]))

    e_c = spark.table("bronze_ecuador_edificios").toPandas()
    lat_c, lon_c = numero(e_c.Latitude), numero(e_c.Longitude)
    manta = pd.DataFrame({"ciudad": "Manta", "id": e_c.Experiment_ID, "lat": lat_c, "lon": lon_c, "dano": e_c.Structural_Damage,
                          "pisos": numero(e_c.No_Floors), "area_m2": numero(e_c.Floor_Area)})
    manta = manta[lon_c.between(-80.80, -80.62) & lat_c.between(-1.03, -0.90)].dropna(subset=["pisos", "area_m2"]).copy()
    manta["vs30"] = vs30_ciudad("Manta", manta.lon, manta.lat)
    manta = completar_edificio(manta)
    edificios_dano = pd.concat([tainan, manta], ignore_index=True)[["ciudad", "id", "lat", "lon", "dano", "severo", "pisos", "altura_m", "area_m2", "Tb", "vs30", "suelo_cod"]]
    subir_pandas(edificios_dano, "silver_dano_edificios",
                 esquema="ciudad string, id string, lat double, lon double, dano string, severo boolean, pisos long, altura_m double, area_m2 double, Tb double, vs30 double, suelo_cod double")
    display(edificios_dano.groupby("ciudad").agg(edificios=("id", "size"), severos=("severo", "mean"), pisos=("pisos", "median"), area=("area_m2", "median"), vs30=("vs30", "median")).round(2))

In [ ]:
if procesar_maestros:
    lst = spark.table("bronze_tainan_lista_registros").toPandas()
    lst["fila"] = numero(lst.fila)
    lst = lst[lst.fila > 1].sort_values(["libro", "fila"]).drop_duplicates("A")
    coord = lst.J.astype(str).str.extract(r"([\d.]+)\D+([\d.]+)").astype(float)
    est_t = pd.DataFrame({"estacion": lst.A.str.strip(), "lat": coord[0], "lon": coord[1],
                          "archivo": lst.K.astype(str).str.extract(r"([0-9A-Za-z]+(?:\.[0-9A-Za-z]+)*\.txt)")[0]}).dropna(subset=["lat", "lon", "archivo"])
    xe, ye = a_utm_tainan(est_t.lon.to_numpy(), est_t.lat.to_numpy())
    xb, yb = a_utm_tainan(tainan.lon.to_numpy(), tainan.lat.to_numpy())
    est_t = est_t[cKDTree(np.column_stack([xb, yb]) / 1000).query(np.column_stack([xe, ye]) / 1000)[0] <= RADIO_TAINAN_KM].reset_index(drop=True)
    est_t["vs30"] = vs30_ciudad("Tainan", est_t.lon, est_t.lat)
    textos_t = (spark.table("bronze_tainan_registros").where(F.col("archivo").isin(est_t.archivo.tolist())).select("archivo", "contenido").toPandas().set_index("archivo").contenido)
    filas_ondas = []
    for f in est_t.itertuples():
        if f.archivo not in textos_t.index or not np.isfinite(f.vs30):
            continue
        texto = textos_t[f.archivo]
        fs = float(re.search(r"SampleRate\(Hz\):\s*([\d.]+)", texto[:800]).group(1))
        a = pd.read_csv(io.StringIO(texto), sep=r"\s+", comment="#", header=None, engine="python").to_numpy(float)
        if a.shape[1] != 4 or len(a) < 20 * fs:
            continue
        filas_ondas.append({"ciudad": "Tainan", "evento": "2016-02-06_meinong_Mw6.4", "archivo_estacion": f.archivo, "estacion": f.estacion, "lat": f.lat, "lon": f.lon,
                            "vs30": f.vs30, "fs": fs, **variables_onda(a[:, 2], a[:, 3], fs)})


    def leer_renac(texto):
        fs = float(re.search(r"muestreo \(Hz\):\s*([0-9.eE+-]+)", texto).group(1))
        return fs, np.array(texto.split("____")[-1].split("\n", 1)[1].split(), float)


    textos_m = spark.table("bronze_manta_registros").select("archivo", "contenido").toPandas().set_index("archivo").contenido
    fs_m, e_m = leer_renac(textos_m[[n for n in textos_m.index if n.startswith("AMNT") and "_E_" in n][0]])
    _, n_m = leer_renac(textos_m[[n for n in textos_m.index if n.startswith("AMNT") and "_N_" in n][0]])
    LON_AMNT, LAT_AMNT = -80.735, -0.941
    filas_ondas.append({"ciudad": "Manta", "evento": "2016-04-16_ecuador_Mw7.8", "archivo_estacion": "AMNT", "estacion": "AMNT", "lat": LAT_AMNT, "lon": LON_AMNT,
                        "vs30": float(vs30_ciudad("Manta", [LON_AMNT], [LAT_AMNT])[0]), "fs": fs_m, **variables_onda(e_m, n_m, fs_m)})
    subir_pandas(pd.DataFrame(filas_ondas), "silver_dano_registros",
                 esquema="ciudad string, evento string, archivo_estacion string, estacion string, lat double, lon double, vs30 double, fs double, pga_geo_g double, pgv_geo_cms double, tp_s double, sa_max_geo_g double, sa_geo array<double>, arias_ms double, d595_s double")

### Suelo no lineal y fragilidad

In [ ]:
if procesar_maestros:
    escribir(spark.table("bronze_bssa14_no_lineal").select(*[F.col(c).cast("double") for c in ("T_s", "f4", "f5")]), "silver_bssa14_no_lineal")

In [ ]:
if procesar_maestros:
    ESTADOS = ["leve", "moderado", "severo", "colapso"]
    FRAGILIDAD_GEM = {
        "adobe": ("MUR-ADO/LWAL-DNO", [
            ( 1, "SA(0.3s)", 0.249, 0.621, 0.918, 1.168, 0.584),
            ( 2, "SA(0.3s)", 0.253, 0.642, 0.987, 1.291, 0.616),
            ( 3, "SA(0.6s)", 0.109, 0.302, 0.487, 0.658, 0.496),
        ]),
        "quincha": ("W-WWD/LWAL-DNO", [
            ( 1, "SA(0.3s)", 0.280, 0.679, 0.985, 1.237, 0.553),
            ( 2, "SA(0.3s)", 0.255, 0.722, 1.118, 1.463, 0.660),
        ]),
        "madera": ("W/LFM-DUL", [
            ( 1, "SA(0.3s)", 0.620, 1.128, 1.637, 2.112, 0.430),
            ( 2, "SA(0.3s)", 0.691, 1.346, 2.001, 2.616, 0.510),
        ]),
        "albanileria_confinada": ("MCF/LWAL-DUL", [
            ( 1, "PGA", 0.565, 0.941, 1.200, 1.400, 0.370),
            ( 2, "SA(0.3s)", 0.427, 1.205, 1.906, 2.535, 0.648),
            ( 3, "SA(0.3s)", 0.458, 1.109, 1.679, 2.181, 0.597),
            ( 4, "SA(0.6s)", 0.223, 0.545, 0.856, 1.144, 0.530),
            ( 5, "SA(0.6s)", 0.261, 0.578, 0.890, 1.181, 0.537),
            ( 6, "SA(0.6s)", 0.327, 0.639, 0.947, 1.236, 0.539),
        ]),
        "concreto_armado": ("CR/LFINF-DUM", [
            ( 5, "SA(0.6s)", 0.305, 0.946, 1.566, 2.144, 0.553),
            ( 6, "SA(0.6s)", 0.380, 1.053, 1.711, 2.326, 0.555),
            ( 7, "SA(1.0s)", 0.227, 0.577, 0.929, 1.262, 0.553),
            ( 8, "SA(1.0s)", 0.290, 0.650, 1.013, 1.357, 0.540),
            ( 9, "SA(1.0s)", 0.363, 0.719, 1.077, 1.413, 0.514),
            (10, "SA(1.0s)", 0.443, 0.790, 1.138, 1.464, 0.501),
        ]),
    }


    def ajustar_gem(contenido):
        d = pd.read_csv(io.StringIO(contenido))
        x = d.iloc[:, 0].to_numpy()
        thetas, betas = [], []
        for col in d.columns[1:]:
            p = d[col].to_numpy()
            m = (p > 0.02) & (p < 0.98)
            pendiente, corte = np.polyfit(np.log(x[m]), norm.ppf(p[m]), 1)
            thetas.append(round(float(np.exp(-corte / pendiente)), 3))
            betas.append(1 / pendiente)
        return d.columns[0].split(" ")[0], thetas, round(float(np.mean(betas)), 3)


    curvas_gem = spark.table("bronze_fragilidad_gem").select("archivo", "contenido").toPandas().set_index("archivo").contenido
    for tip, (tax, filas) in FRAGILIDAD_GEM.items():
        recalculadas = []
        for p, im, *th, b in filas:
            archivo = f"{tax.replace('/', '_')}_H{p}.csv"
            if archivo in curvas_gem.index:
                im, th, b = ajustar_gem(curvas_gem[archivo])
            recalculadas.append((p, im, *th, b))
        FRAGILIDAD_GEM[tip] = (tax, recalculadas)
    FUENTE_FRAGILIDAD = ("Martins L, Silva V (2021). Development of a fragility and vulnerability model for global seismic risk "
                         "analyses. Bulletin of Earthquake Engineering. doi:10.1007/s10518-020-00885-1 | "
                         "github.com/lmartins88/global_fragility_vulnerability (CC BY-SA 4.0)")

    frag_pdf = pd.DataFrame([{"tipologia": t, "taxonomia_gem": tax, "pisos_gem": p, "im": im,
                              **dict(zip([f"theta_{e}" for e in ESTADOS], th)), "beta": b, "fuente": FUENTE_FRAGILIDAD}
                             for t, (tax, filas) in FRAGILIDAD_GEM.items() for p, im, *th, b in filas])
    subir_pandas(frag_pdf, "silver_fragilidad_parametros")

## 10. Verificación

In [ ]:
spark.table("silver_ondas_registros_cuarentena").groupBy("evento", "motivo_rechazo").count().orderBy("evento").show(truncate=False)
display(inventario("silver_"))
display(spark.table("silver_calidad_log").groupBy("tabla", "accion").agg(F.sum("n_filas").alias("filas")).orderBy("tabla").toPandas())